In [2]:
import numpy as np
import argparse
import sys

from aie.iron import Kernel, ObjectFifo, Program, Runtime, Worker
from aie.iron.placers import SequentialPlacer
from aie.iron.device import NPU1Col2, NPU1Col1, NPU2
from aie.iron.controlflow import range_

In [7]:
def build_elementwise_robo(device, size):
    value_type = np.half
    param_type = np.ndarray[(1, ), np.dtype[np.int32]]
    input_type = np.ndarray[(3 * size, ), np.dtype[value_type]]
    output_type = np.ndarray[(3 * size, ), np.dtype[value_type]]

    kernel_fn = Kernel(
        "elementwise_inc",
        "elementwise_incr.o",
        [param_type, input_type, output_type]
    )

    of_in_param = ObjectFifo(param_type, name="param_input_fifo")
    of_in_data = ObjectFifo(input_type, name="data_input_fifo")
    of_out = ObjectFifo(output_type, name="output_fifo")

    def core_fn(of_param, of_data, of_out, kernel):
        p = of_param.acquire(1)
        o = of_out.acquire(1)
        i = of_data.acquire(1)
        kernel(p, i, o)
        of_data.release(1)
        of_out.release(1)
        of_param.release(1)

    worker = Worker(core_fn, [of_in_param.cons(), of_in_data.cons(), of_out.prod(), kernel_fn])

    rt = Runtime()

    with rt.sequence(param_type, input_type, output_type) as (p, i, o):
        rt.start(worker)
        rt.fill(of_in_param.prod(), p)
        rt.fill(of_in_data.prod(), i)
        rt.drain(of_out.cons(), o, wait=True)

    program = Program(device, rt)

    return program.resolve_program(SequentialPlacer())
    

In [8]:
with open("ewi.mlir", 'w') as f:
    print(build_elementwise_robo(NPU1Col1(), 96), file=f)

In [12]:
dir(build_elementwise_robo(NPU2(), 1024).operation)

['_CAPICreate',
 '_CAPIPtr',
 '__class__',
 '__delattr__',
 '__dir__',
 '__doc__',
 '__eq__',
 '__format__',
 '__ge__',
 '__getattribute__',
 '__getstate__',
 '__gt__',
 '__hash__',
 '__init__',
 '__init_subclass__',
 '__le__',
 '__lt__',
 '__module__',
 '__ne__',
 '__new__',
 '__reduce__',
 '__reduce_ex__',
 '__repr__',
 '__setattr__',
 '__sizeof__',
 '__str__',
 '__subclasshook__',
 'attributes',
 'clone',
 'context',
 'create',
 'detach_from_parent',
 'erase',
 'get_asm',
 'location',
 'move_after',
 'move_before',
 'name',
 'operands',
 'operation',
 'opview',
 'parent',
 'parse',
 'print',
 'regions',
 'result',
 'results',
 'successors',
 'verify',
 'walk',
 'write_bytecode']

In [6]:
print(build_elementwise_robo(NPU1Col1(), 1024))

NameError: name 'NPU1Col1' is not defined